# 📚 [5주차] 데이터 분석 전처리반(판다스) 복습과제
### 작성자: 이창현
---
### 📌 복습 키워드
- 결측치
- MICE (Multiple Imputation by Chained Equations)
- Simple Imputer
- Interpolate
- 보간법

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'   # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print('라이브러리 로드 완료 ✅')

---
## 1️⃣ 결측치 (Missing Value) 기본 개념

| 표현 | 의미 |
|------|------|
| `NaN` | Not a Number - 수치형 결측 |
| `None` | 파이썬 기본 결측 |
| `NaT` | Not a Time - 날짜형 결측 |

### 주요 결측치 확인 메서드
| 메서드 | 설명 |
|--------|------|
| `isnull()` / `isna()` | 결측치 여부 확인 (True/False) |
| `notnull()` | 결측치 아닌 여부 확인 |
| `isnull().sum()` | 컬럼별 결측치 개수 |
| `isnull().mean()` | 컬럼별 결측치 비율 |

In [ ]:
# 예제 데이터 생성
df = pd.DataFrame({
    'age'     : [25, 32, np.nan, 28, 55, np.nan, 60, 22, 47, 35],
    'income'  : [3000, np.nan, 6000, 3200, np.nan, 5000, 8000, np.nan, 6500, 4800],
    'score'   : [80, 75, 90, np.nan, 70, 88, np.nan, 92, 78, 83],
    'grade'   : ['A', 'B', None, 'A', 'C', None, 'B', 'A', None, 'B']
})

print('📌 원본 데이터')
print(df)
print()

print('📌 결측치 개수')
print(df.isnull().sum())
print()

print('📌 결측치 비율')
print(round(df.isnull().mean() * 100, 1).astype(str) + '%')

In [ ]:
# 결측치 시각화
fig, ax = plt.subplots(figsize=(8, 4))

missing = df.isnull()
ax.imshow(missing, cmap='Reds', aspect='auto')
ax.set_xticks(range(len(df.columns)))
ax.set_xticklabels(df.columns)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(range(len(df)))
ax.set_title('결측치 히트맵 (빨강 = 결측)')
plt.tight_layout()
plt.show()

---
## 2️⃣ Simple Imputer

- **개념**: sklearn의 단순 결측치 대체 클래스
- **특징**: 컬럼 단위로 단일 대표값으로 일괄 대체

| strategy | 설명 | 적합한 자료형 |
|----------|------|---------------|
| `mean` | 평균으로 대체 | 수치형 |
| `median` | 중앙값으로 대체 | 수치형 (이상치 있을 때) |
| `most_frequent` | 최빈값으로 대체 | 범주형 / 수치형 |
| `constant` | 지정한 상수로 대체 | 모든 자료형 |

In [ ]:
# ① mean 전략 - 평균으로 대체
imputer_mean = SimpleImputer(strategy='mean')
df_mean = df[['age', 'income', 'score']].copy()
df_mean_imputed = pd.DataFrame(
    imputer_mean.fit_transform(df_mean),
    columns=df_mean.columns
)
print('📌 SimpleImputer (strategy=mean)')
print(df_mean_imputed)
print()

In [ ]:
# ② median 전략 - 중앙값으로 대체
imputer_median = SimpleImputer(strategy='median')
df_median_imputed = pd.DataFrame(
    imputer_median.fit_transform(df_mean),
    columns=df_mean.columns
)
print('📌 SimpleImputer (strategy=median)')
print(df_median_imputed)
print()

In [ ]:
# ③ most_frequent 전략 - 범주형 결측치
imputer_freq = SimpleImputer(strategy='most_frequent')
df_grade = df[['grade']].copy()
df_grade_imputed = pd.DataFrame(
    imputer_freq.fit_transform(df_grade),
    columns=['grade']
)
print('📌 SimpleImputer (strategy=most_frequent) - 범주형')
print(df_grade_imputed)
print()

# ④ constant 전략 - 고정값으로 대체
imputer_const = SimpleImputer(strategy='constant', fill_value=0)
df_const_imputed = pd.DataFrame(
    imputer_const.fit_transform(df_mean),
    columns=df_mean.columns
)
print('📌 SimpleImputer (strategy=constant, fill_value=0)')
print(df_const_imputed)

---
## 3️⃣ Interpolate (보간법)

- **개념**: 앞뒤 데이터를 활용해 결측값을 수학적으로 추정
- **특징**: 시계열 데이터나 순서가 있는 데이터에 적합

| method | 설명 |
|--------|------|
| `linear` | 선형 보간 (기본값) - 앞뒤 값의 중간 |
| `ffill` | 앞 값으로 채우기 (forward fill) |
| `bfill` | 뒤 값으로 채우기 (backward fill) |
| `polynomial` | 다항식 보간 |
| `spline` | 스플라인 보간 |

In [ ]:
# 시계열 형태의 예제 데이터
ts_data = pd.Series(
    [10, np.nan, 30, np.nan, np.nan, 60, 70, np.nan, 90, 100],
    index=pd.date_range('2024-01-01', periods=10, freq='D')
)

print('📌 원본 데이터')
print(ts_data)
print()

# ① 선형 보간
ts_linear = ts_data.interpolate(method='linear')
print('📌 linear 보간')
print(ts_linear)
print()

In [ ]:
# ② ffill / bfill 비교
ts_ffill = ts_data.ffill()
ts_bfill = ts_data.bfill()

print('📌 ffill (앞 값으로 채우기)')
print(ts_ffill.values)
print()
print('📌 bfill (뒤 값으로 채우기)')
print(ts_bfill.values)

In [ ]:
# ③ 보간법 비교 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('보간법 비교', fontsize=14)

methods = [
    ('원본', ts_data),
    ('linear', ts_data.interpolate(method='linear')),
    ('ffill',  ts_data.ffill()),
    ('bfill',  ts_data.bfill())
]

for ax, (title, data) in zip(axes.flatten(), methods):
    ax.plot(data, marker='o', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('날짜')
    ax.set_ylabel('값')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4️⃣ MICE (Multiple Imputation by Chained Equations)

- **개념**: 여러 변수 간의 관계를 활용해 결측치를 반복적으로 추정
- **sklearn 구현**: `IterativeImputer`
- **Simple Imputer와 차이점**:

| 구분 | Simple Imputer | MICE (IterativeImputer) |
|------|---------------|-------------------------|
| 방식 | 단일 통계값 대체 | 다른 변수와의 관계를 모델링 |
| 정확도 | 낮음 | 높음 |
| 속도 | 빠름 | 느림 |
| 적합한 경우 | 변수 간 관계 없을 때 | 변수 간 상관관계 있을 때 |

### MICE 동작 원리
```
① 초기값 설정 (평균으로 임시 대체)
② 각 결측 변수를 나머지 변수로 회귀 모델 학습
③ 학습된 모델로 결측치 예측 및 대체
④ ②~③ 반복 (수렴할 때까지)
```

In [ ]:
# 변수 간 상관관계가 있는 예제 데이터
np.random.seed(42)
n = 50

age    = np.random.randint(20, 60, n).astype(float)
income = age * 100 + np.random.normal(0, 500, n)   # age와 양의 상관관계
score  = income * 0.01 + np.random.normal(0, 5, n) # income과 양의 상관관계

df_mice = pd.DataFrame({
    'age'   : age,
    'income': income,
    'score' : score
})

# 결측치 임의 삽입 (20%)
for col in ['income', 'score']:
    missing_idx = np.random.choice(df_mice.index, size=10, replace=False)
    df_mice.loc[missing_idx, col] = np.nan

print('📌 결측치 포함 데이터')
print(df_mice.isnull().sum())
print(df_mice.head(10))

In [ ]:
# Simple Imputer (mean) 적용
simple_imp = SimpleImputer(strategy='mean')
df_simple = pd.DataFrame(
    simple_imp.fit_transform(df_mice),
    columns=df_mice.columns
)

# MICE (IterativeImputer) 적용
mice_imp = IterativeImputer(
    max_iter=10,    # 반복 횟수
    random_state=42
)
df_mice_imputed = pd.DataFrame(
    mice_imp.fit_transform(df_mice),
    columns=df_mice.columns
)

print('📌 Simple Imputer 결과 (처음 5행)')
print(df_simple.head())
print()
print('📌 MICE 결과 (처음 5행)')
print(df_mice_imputed.head())

In [ ]:
# Simple Imputer vs MICE 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Simple Imputer vs MICE 비교 (age vs income)', fontsize=13)

# 결측치 위치 마스크
missing_mask = df_mice['income'].isnull()

# Simple Imputer
axes[0].scatter(
    df_simple.loc[~missing_mask, 'age'],
    df_simple.loc[~missing_mask, 'income'],
    label='원본', alpha=0.7, color='steelblue'
)
axes[0].scatter(
    df_simple.loc[missing_mask, 'age'],
    df_simple.loc[missing_mask, 'income'],
    label='대체값', alpha=0.9, color='red', marker='X', s=100
)
axes[0].set_title('Simple Imputer (mean)')
axes[0].set_xlabel('age')
axes[0].set_ylabel('income')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MICE
axes[1].scatter(
    df_mice_imputed.loc[~missing_mask, 'age'],
    df_mice_imputed.loc[~missing_mask, 'income'],
    label='원본', alpha=0.7, color='steelblue'
)
axes[1].scatter(
    df_mice_imputed.loc[missing_mask, 'age'],
    df_mice_imputed.loc[missing_mask, 'income'],
    label='대체값', alpha=0.9, color='red', marker='X', s=100
)
axes[1].set_title('MICE (IterativeImputer)')
axes[1].set_xlabel('age')
axes[1].set_ylabel('income')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 MICE는 age와 income의 상관관계를 반영해서 대체값이 더 자연스러움!')

---
## 5️⃣ 보간법 종합 비교

| 방법 | 클래스/메서드 | 특징 | 적합한 상황 |
|------|--------------|------|-------------|
| **Simple Imputer** (mean) | `SimpleImputer(strategy='mean')` | 빠르고 간단 | MCAR, 변수 간 관계 없음 |
| **Simple Imputer** (median) | `SimpleImputer(strategy='median')` | 이상치에 강건 | 이상치 많은 수치형 |
| **Simple Imputer** (most_frequent) | `SimpleImputer(strategy='most_frequent')` | 범주형 대체 | 범주형 변수 |
| **Interpolate** (linear) | `interpolate(method='linear')` | 앞뒤 값 선형 추정 | 시계열, 순서 있는 데이터 |
| **Interpolate** (ffill/bfill) | `ffill()` / `bfill()` | 인접값 그대로 사용 | 시계열 |
| **MICE** | `IterativeImputer` | 변수 간 관계 반영 | MAR, 변수 간 상관관계 있음 |

In [ ]:
# 전체 방법 한눈에 비교
df_compare = df[['age', 'income', 'score']].copy()

results = {
    'original'     : df_compare,
    'simple_mean'  : pd.DataFrame(
                        SimpleImputer(strategy='mean').fit_transform(df_compare),
                        columns=df_compare.columns),
    'simple_median': pd.DataFrame(
                        SimpleImputer(strategy='median').fit_transform(df_compare),
                        columns=df_compare.columns),
    'mice'         : pd.DataFrame(
                        IterativeImputer(random_state=42).fit_transform(df_compare),
                        columns=df_compare.columns),
    'interpolate'  : df_compare.interpolate(method='linear')
}

print('📊 방법별 income 컬럼 비교')
comparison = pd.DataFrame({
    method: data['income']
    for method, data in results.items()
})
print(round(comparison, 2))
print()
print('💡 결측 위치:', df_compare[df_compare['income'].isnull()].index.tolist())

---
## 📝 전체 요약 정리

| 키워드 | 핵심 내용 |
|--------|----------|
| **결측치** | NaN/None/NaT → isnull().sum() 으로 확인 |
| **Simple Imputer** | 단일 통계값(평균/중앙값/최빈값)으로 빠르게 대체 |
| **Interpolate** | 앞뒤 데이터 활용 → 시계열에 적합 (linear/ffill/bfill) |
| **MICE** | 변수 간 관계 반영 → 정확도 높지만 느림 (IterativeImputer) |
| **보간법 선택 기준** | MCAR→Simple, 시계열→Interpolate, 변수 간 관계→MICE |